<a href="https://colab.research.google.com/github/loganjoslin/ENPH353_Lab2/blob/main/LAB2_Logan_J.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [57]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow

In [58]:
## \brief Draws a red circle on an image.
# \param frame image being modified
# \param location centre of the circle to be superimposed
# \param radius radius of the circle to be superimposed

def superimpose_red_circle(frame, location, radius):
  for x in range(location[0] - radius, location[0] + radius):
    for y in range(location[1] - radius, location[1] + radius):
      pointInCirle = (x - location[0])**2 + (y - location[1])**2 <= radius**2
      pointInImage = (x >= 0) and (y>=0) and (x < width) and (y < height)
      if pointInCirle and pointInImage:
        frame[y, x] = [0, 0, 255]

In [59]:
## \file
# \brief Superimposes a red circle that tracks the road in the inputted video

# Magic numbers
RED_SPLOTCH_RADIUS = 10
LINE_ANALYSIS_POSITION = 0.9
ROAD_DARKNESS_THRESHOLD = 120

# Input and output file locations
input = "/content/raw_video_feed.mp4"
output = "/content/output_video_feed.mp4"

# Open the input video
video = cv.VideoCapture(input)

if not video.isOpened():
    raise FileNotFoundError("Could not open the input video.")

# Read and print video info
width = int(video.get(cv.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv.CAP_PROP_FRAME_HEIGHT))
fps = video.get(cv.CAP_PROP_FPS)
frame_count = int(video.get(cv.CAP_PROP_FRAME_COUNT))

# Create output video with specs
fourcc = cv.VideoWriter_fourcc(*"mp4v") # method of video compression
outputVideo = cv.VideoWriter(
    output, # file path
    fourcc,
    fps, # same as input
    (width, height), # same as input
    isColor=True # displaying the original coloured image
)

if not outputVideo.isOpened():
    raise RuntimeError("Could not create the output video.")

while True:

  success, frame = video.read()
  if not success:
      break # leave loop when video is over, or if there is a reading failure

  # 1. convert to grayscale, then binary -> less data per frame -> faster to process
  grayFrame = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
  usedThreshold, binaryFrame = cv.threshold(
      grayFrame,
      ROAD_DARKNESS_THRESHOLD, # threshold value (gets returned in usedThreshold)
      255, # below threshold, go to this value
      cv.THRESH_BINARY_INV # make the dark pixels WHITE
  )

  # 2. identify the avg "white" position at chosen row; add a red splotchA
  chosenRow = int(LINE_ANALYSIS_POSITION*height)
  allPossibleXPositions = np.arange(width)
  allWhiteXPositions = np.where(binaryFrame[chosenRow, :] == 255, allPossibleXPositions, -1)
  allWhiteXPositions= allWhiteXPositions[allWhiteXPositions != -1]
  if len(allWhiteXPositions) > 0:
    avgWhitePosition = int(np.sum(allWhiteXPositions) / len(allWhiteXPositions))
    superimpose_red_circle(frame, (avgWhitePosition, chosenRow), RED_SPLOTCH_RADIUS)


  outputVideo.write(frame)

video.release()
outputVideo.release()



In [60]:
from IPython.display import Video, display

# Convert output video to browser friendly playable format, then display it
!ffmpeg -y -loglevel error -i /content/output_video_feed.mp4 -c:v libx264 -pix_fmt yuv420p /content/output_video_browser.mp4
display(Video("/content/output_video_browser.mp4", embed=True))